# One acquisition loop, five versions
Each cell below runs one version of the loop on a simulated sensor and a simulated stage, with the same settings as the "Short run" button of the app.
You see where the time goes, and the checks on the file. No uploads, no window. The runs take 2 to 3 minutes in all.

## 1. Get the code
This cell installs numpy and matplotlib, if they are missing, and fetches the demo from the tutorial repository.

In [ ]:
import os
import subprocess
import sys
from pathlib import Path

REPO_URL = "https://github.com/matteolopreti/sensors2026-tutorial"  # placeholder: the address of the public repository
LOCAL_DEMO = os.environ.get("SENSORS_DEMO_DIR")  # set only by the test of this notebook: it uses a local copy instead

if LOCAL_DEMO:
    demo_dir = Path(LOCAL_DEMO)
else:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "numpy", "matplotlib"], check=True)
    if not Path("repo").exists():
        subprocess.run(["git", "clone", "--depth", "1", REPO_URL, "repo"], check=True)
    demo_dir = Path("repo/demo").resolve()
sys.path.insert(0, str(demo_dir))
print("demo code in", demo_dir)

## 2. Set up one short run
A short run is 1 cycle with a hold of 0.3 s, at the default depth and speed. It takes about 21 s (28 s for V0).
The function `short_run` is all the glue there is: the demo itself comes from the repository.

In [ ]:
import numpy as np

import app
import metrics
import settings
import timeline

SHORT_RUN = {**settings.defaults(), **settings.SHORT_RUN}  # what the "Short run" button of the app sets
gap_by_version = {}


def short_run(version, seed=1):
    """Run one version on the simulated rig until it ends. The log file goes to the folder `out`."""
    session = app.Session(version, SHORT_RUN, Path("out"), seed=seed)
    session.start()
    session.join()
    if session.error:
        raise session.error
    return session


def one_gap_in_100(row_times):
    """The handout recipe: sort the gaps from the longest; read gap number ceil(number of gaps / 100). None below 100 gaps.
    The app uses the same function (metrics.one_gap_in_100), so the notebook and the app give the same number."""
    gaps_ms = np.diff(row_times) * 1e3
    if len(gaps_ms) < metrics.GAPS_FOR_1_IN_100:
        return None
    return -(-len(gaps_ms) // 100), len(gaps_ms), metrics.one_gap_in_100(gaps_ms)


def show(version):
    """Run a version, then show where its time goes, the checks on its file, and 1 gap in 100."""
    session = short_run(version)
    row_times = np.loadtxt(session.rig.disk.path, delimiter=",", skiprows=1, usecols=0, ndmin=1)  # the first column of the file
    window = timeline.window_ending_at(version, session.t_end)
    display(timeline.timeline_figure(version, session.recorder.spans(since=window[0]), window, row_times))
    print(f"{len(row_times)} rows in the file {session.rig.disk.path}")
    print("\n".join(app.check_sentences(session)))
    found = one_gap_in_100(row_times)
    gap_by_version[version] = found and found[2]
    if found is None:
        print("Fewer than 100 gaps: too few rows to read 1 gap in 100.")
    else:
        place, n_gaps, gap_ms = found
        print(f"Longest first, gap number {place} of {n_gaps} is {gap_ms:.0f} ms. So 1 gap in 100 is longer than {gap_ms:.0f} ms.")

## 3. V0: the script the paper warns against
It moves, sleeps 2 s, and writes one row. Most of the time it waits, so few rows reach the file.

In [ ]:
show("V0")

## 4. V1: one loop
HO-01, step 1. Read the last line: how long is 1 gap in 100? The app shows the same number.

In [ ]:
show("V1")

## 5. V2: the version with the queue
HO-01, step 2. Read 1 gap in 100 again. Then look at the picture: which part has moved out of the main loop?

In [ ]:
show("V2")

## 6. V3: rows with the time of the sensor frame
Each row now has the time the computer read the frame, and the frame number. Frames that never reach the file now show up in the file.

In [ ]:
show("V3")

## 7. V4: the loop looks twice per frame
It wakes up twice for each sensor frame, and writes a row only for a new frame.

In [ ]:
show("V4")

## 8. Your numbers
These are the numbers of this computer, in this run. Another computer, or another run, gives other numbers: the code was the same.

In [ ]:
import platform

for version, gap in gap_by_version.items():
    print(version, "too few rows to read it" if gap is None else f"1 gap in 100 is longer than {gap:.0f} ms")
print()
print(platform.platform(), "| Python", platform.python_version(), "| processors:", os.cpu_count())

## 9. Try it again
Run V2 once more with the next cell, and compare 1 gap in 100 with the first run. Change the version name to try another one.

In [ ]:
show("V2")